
## Projeto Final – Machine Learning II
### Previsão de Inadimplência de Clientes
### Márcia Aparecida Rodrigues de Sousa

### 1. Introdução

Este projeto tem como objetivo aplicar técnicas de **Machine Learning** para apoiar decisões de crédito e gestão de risco em instituições financeiras.  
O foco principal é prever se um cliente irá se tornar inadimplente no próximo período de faturamento, utilizando modelos de **classificação supervisionada**.  
A variável alvo é binária (**inadimplente** ou **não inadimplente**), caracterizando um problema clássico de classificação com alto valor estratégico para o setor financeiro.

Além da abordagem supervisionada, também foram explorados **modelos não supervisionados (clustering)**, como K-Means, Agglomerative Clustering e DBSCAN.  
Esses métodos permitem identificar **segmentos de clientes com características semelhantes**, fornecendo insights adicionais para estratégias de marketing, análise de perfil e gestão de risco.

Portanto, o projeto combina:
- **Modelos supervisionados**: para previsão direta de inadimplência.  
- **Modelos não supervisionados**: para segmentação e análise exploratória de clientes.  

Essa dupla abordagem amplia o valor da solução, oferecendo tanto previsões precisas quanto insights estratégicos.



### 2. Importação de Bibliotecas

Nesta etapa são importadas as bibliotecas necessárias para o desenvolvimento do projeto.  
Foram utilizadas bibliotecas para:
- **Manipulação de dados**: Pandas e NumPy.  
- **Visualização**: Matplotlib e Seaborn.  
- **Modelagem supervisionada**: Regressão Logística, Random Forest, SVM, Gradient Boosting e AdaBoost.  
- **Modelagem não supervisionada**: K-Means, Agglomerative Clustering e DBSCAN.  
- **Pré-processamento**: StandardScaler e train_test_split.  
- **Avaliação de modelos**: métricas de classificação e métricas de clusterização.  
- **Otimização de hiperparâmetros**: GridSearchCV.




In [ ]:
# Manipulação de dados
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Pré-processamento
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler

# Modelos supervisionados
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC

# Modelos não supervisionados
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN

# Métricas de avaliação - classificação
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    recall_score,
    f1_score,
    roc_auc_score
)

# Métricas de avaliação - clustering
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)



In [ ]:
!pip install seaborn

In [ ]:
import seaborn as sns


### 3. Definição do Problema de Negócio

O problema de negócio consiste em prever se um cliente irá se tornar inadimplente no próximo período de faturamento.  
A variável alvo é binária (**inadimplente** ou **não inadimplente**).  

Trata-se de um problema clássico de **classificação supervisionada**, com alto valor estratégico para o setor financeiro.  
A correta identificação de clientes inadimplentes permite reduzir riscos de crédito, otimizar políticas de concessão e apoiar decisões estratégicas em instituições financeiras.

Além da abordagem supervisionada, também foram aplicados **modelos não supervisionados (clustering)**, como K-Means, Agglomerative Clustering e DBSCAN.  
Esses métodos não utilizam a variável alvo, mas permitem identificar **segmentos de clientes com características semelhantes**, fornecendo insights adicionais para estratégias de marketing, análise de perfil e gestão de risco.




### 4. Carregamento do Dataset

In [ ]:
# 1. Carregar o dataset
url = "https://raw.githubusercontent.com/Marcia520/ProjetoFinal_ML_II/main/credit_card_default.csv"
df = pd.read_csv(url)


df = df.rename(columns={"default payment_next_month": "inadimplente"})

# 3. Separar features e variável alvo
X = df.drop("inadimplente", axis=1)
y = df["inadimplente"]

# 4. Divisão em treino e teste com estratificação
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# 5. Padronização das variáveis numéricas
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Conferindo dimensões
print("Treino:", X_train.shape, "Teste:", X_test.shape)
print("Distribuição da variável alvo no treino:", y_train.value_counts(normalize=True))






Treino: (21000, 24) Teste: (9000, 24)
Distribuição da variável alvo no treino: inadimplente
0    0.77881
1    0.22119
Name: proportion, dtype: float64


### 5. Descrição do Dataset

O dataset utilizado foi obtido na plataforma Kaggle, conhecido como Credit Card Default Dataset. Ele contém aproximadamente 30.000 registros, atendendo ao requisito mínimo de volume de dados para o projeto.

As variáveis incluem:

 - informações demográficas (sexo, escolaridade, estado civil);
 - limite de crédito concedido;
 - histórico de pagamentos anteriores;
 - valores faturados e pagos em períodos anteriores.

A variável alvo indica se o cliente entrou em inadimplência no mês seguinte, permitindo a construção de modelos preditivos com base em dados históricos.

### 6. Pré-processamento

Foi realizada uma análise exploratória dos dados para compreender sua estrutura, tipos de variáveis e distribuição da variável alvo.  
Observou-se que a classe de inadimplência apresenta **desbalanceamento**, o que influenciou diretamente a escolha das métricas de avaliação.

O pré-processamento foi dividido em dois fluxos:

### Modelos Supervisionados
- Separação entre variáveis explicativas (features) e variável alvo.  
- Divisão do dataset em conjuntos de treino e teste, utilizando **estratificação** para preservar a proporção das classes.  
- Padronização das variáveis numéricas por meio de **StandardScaler**, especialmente para modelos sensíveis à escala (SVM, Regressão Logística).

### Modelos Não Supervisionados
- Utilização apenas das **features** (sem variável alvo).  
- Padronização das variáveis numéricas para evitar distorções em algoritmos baseados em distância (K-Means, DBSCAN, Agglomerative).  
- Possível redução de dimensionalidade (PCA ou t-SNE) para visualização dos clusters.

Essas transformações garantem maior estabilidade no treinamento dos modelos e evitam vieses decorrentes de escalas diferentes entre as variáveis.


In [ ]:
# -------------------------
# Supervisionados
# -------------------------

# Separação entre features e variável alvo
X = df.drop("inadimplente", axis=1)
y = df["inadimplente"]

# Divisão em treino e teste com estratificação
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Padronização das variáveis numéricas
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# -------------------------
# Não supervisionados
# -------------------------

# Apenas features (sem variável alvo)
X_unsupervised = df.drop("inadimplente", axis=1)

# Padronização para clustering
scaler_unsup = StandardScaler()
X_unsupervised_scaled = scaler_unsup.fit_transform(X_unsupervised)





In [ ]:
df.columns

Index(['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0',
       'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2',
       'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6',
       'inadimplente'],
      dtype='object')

### 7. Tratamento e Transformação dos Dados

Após o pré-processamento inicial, os dados foram preparados de forma distinta para os dois enfoques do projeto:

### Modelos Supervisionados
- Divisão em treino e teste com **estratificação**, garantindo que a proporção da classe alvo (inadimplente vs. não inadimplente) seja preservada.  
- Padronização das variáveis numéricas com **StandardScaler**, essencial para algoritmos sensíveis à escala (SVM, Regressão Logística).  
- Verificação das dimensões dos conjuntos para assegurar consistência.

### Modelos Não Supervisionados
- Utilização apenas das **features** (sem variável alvo).  
- Padronização das variáveis numéricas para evitar distorções em algoritmos baseados em distância (K-Means, DBSCAN, Agglomerative).  
- Preparação de um dataset escalado específico para clustering.

Essas transformações garantem que os dados estejam prontos para os dois tipos de abordagem: previsão supervisionada e segmentação não supervisionada.


In [ ]:
# -------------------------
# supervisionados
# -------------------------

# Divisão treino/teste com estratificação
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Padronização das variáveis numéricas
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Verificando dimensões dos conjuntos
print("Treino:", X_train.shape, "Teste:", X_test.shape)

# -------------------------
# Não supervisionados
# -------------------------

# Apenas features (sem variável alvo)
X_unsupervised = df.drop("inadimplente", axis=1)

# Padronização para clustering
scaler_unsup = StandardScaler()
X_unsupervised_scaled = scaler_unsup.fit_transform(X_unsupervised)

# Verificando dimensões do dataset para clustering
print("Dataset não supervisionado:", X_unsupervised_scaled.shape)


Treino: (21000, 24) Teste: (9000, 24)
Dataset não supervisionado: (30000, 24)


### 8. Métricas de Avaliação

Devido ao desbalanceamento das classes e à natureza do problema de risco de crédito, a métrica **Recall** foi priorizada.  
Em cenários de inadimplência, é mais crítico identificar corretamente clientes inadimplentes do que classificar erroneamente um cliente adimplente como risco.

### Modelos Supervisionados
As métricas utilizadas foram:
- **Recall**: prioridade para identificar inadimplentes.  
- **F1-Score**: equilíbrio entre precisão e recall.  
- **ROC-AUC**: capacidade discriminativa dos modelos.  

A utilização de múltiplas métricas permite uma avaliação mais robusta e alinhada ao contexto de negócio.

### Modelos Não Supervisionados
Como não há variável alvo, foram utilizadas métricas de qualidade de clusterização:
- **Silhouette Score**: mede a separação entre clusters (quanto mais próximo de 1, melhor).  
- **Davies-Bouldin Index**: avalia a similaridade entre clusters (quanto menor, melhor).  
- **Calinski-Harabasz Index**: mede a dispersão intra e inter-clusters (quanto maior, melhor).  

Essas métricas permitem avaliar a coerência e a separação dos grupos formados pelos algoritmos de clustering.



In [ ]:
# 1. Divisão em treino e teste
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 2. Normalização
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Definição e treino do modelo
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)


In [ ]:
# Exemplo de avaliação para um modelo supervisionado
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:,1]

print("Recall:", recall_score(y_test, y_pred))
print("F1-Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))


In [10]:
# Exemplo de avaliação para um modelo não supervisionado (K-Means)
kmeans = KMeans(n_clusters=2, random_state=42)
labels_kmeans = kmeans.fit_predict(X_unsupervised_scaled)

print("Silhouette:", silhouette_score(X_unsupervised_scaled, labels_kmeans))
print("Davies-Bouldin:", davies_bouldin_score(X_unsupervised_scaled, labels_kmeans))
print("Calinski-Harabasz:", calinski_harabasz_score(X_unsupervised_scaled, labels_kmeans))


Silhouette: 0.33345430270331083
Davies-Bouldin: 1.5855510133896384
Calinski-Harabasz: 6090.854409802133


### 9. Modelagem com Machine Learning

Foram utilizados modelos de naturezas distintas, com o objetivo de comparar desempenho e obter insights complementares.

### Modelos Supervisionados
O foco principal foi prever inadimplência:
- **9.1 Regressão Logística**: utilizada como modelo baseline, por ser amplamente aplicada em problemas de crédito, apresentar boa interpretabilidade e servir como referência inicial de desempenho.  
- **9.2 Random Forest**: algoritmo baseado em conjuntos de árvores de decisão, capaz de capturar relações não lineares e interações complexas entre variáveis.  
- **9.3 Support Vector Machine (SVM)**: bom para separação em alta dimensionalidade, sensível à escala dos dados.  
- **9.4 Gradient Boosting**: modelo de boosting sequencial, corrige erros iterativamente e costuma ter alta performance.  
- **9.5 AdaBoost**: boosting com pesos adaptativos, útil para explorar diferentes combinações de classificadores fracos.

### Modelos Não Supervisionados
Além da previsão, foram aplicados algoritmos de clustering para identificar segmentos de clientes:
- **K-Means**: particiona os dados em grupos definidos previamente.  
- **Agglomerative Clustering**: método hierárquico que agrupa instâncias de forma progressiva.  
- **DBSCAN**: identifica clusters de densidade, permitindo detectar outliers.  

Ambos os enfoques foram treinados e avaliados utilizando os mesmos dados (com ou sem variável alvo), garantindo consistência na comparação.


### Supervisionados

In [ ]:
# Regressão Logística
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_scaled, y_train)
y_pred_log = log_reg.predict(X_test_scaled)

# Random Forest
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# SVM
svm = SVC(kernel="rbf", probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)
y_pred_svm = svm.predict(X_test_scaled)

# Gradient Boosting
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

# AdaBoost
ada = AdaBoostClassifier(random_state=42)
ada.fit(X_train, y_train)
y_pred_ada = ada.predict(X_test)


### Não Supervisionados

In [ ]:
# K-Means
kmeans = KMeans(n_clusters=2, random_state=42)
labels_kmeans = kmeans.fit_predict(X_unsupervised_scaled)

# Agglomerative Clustering
agg = AgglomerativeClustering(n_clusters=2)
labels_agg = agg.fit_predict(X_unsupervised_scaled)

# DBSCAN
dbscan = DBSCAN(eps=3, min_samples=5)
labels_dbscan = dbscan.fit_predict(X_unsupervised_scaled)


### 10. Otimização de Hiperparâmetros

Para melhorar o desempenho dos modelos, foi utilizada a técnica de **GridSearchCV**, que realiza busca exaustiva em combinações de parâmetros.



In [ ]:
# GridSearch para Regressão Logística
param_grid_log = {"C":[0.01,0.1,1,10], "penalty":["l2"]}
grid_log = GridSearchCV(LogisticRegression(random_state=42), param_grid_log, cv=5, scoring="recall")
grid_log.fit(X_train_scaled, y_train)

# GridSearch para Random Forest
param_grid_rf = {"n_estimators":[100,200], "max_depth":[None,10,20]}
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=5, scoring="recall")
grid_rf.fit(X_train, y_train)

print("Melhores parâmetros Logística:", grid_log.best_params_)
print("Melhores parâmetros RF:", grid_rf.best_params_)



### 11. Comparação dos Modelos

Os modelos supervisionados foram comparados utilizando Recall, F1-Score e ROC-AUC.



In [ ]:
results = {
    "Modelo":["Logística","Random Forest","SVM","Gradient Boosting","AdaBoost"],
    "Recall":[recall_score(y_test,y_pred_log),
              recall_score(y_test,y_pred_rf),
              recall_score(y_test,y_pred_svm),
              recall_score(y_test,y_pred_gb),
              recall_score(y_test,y_pred_ada)],
    "F1":[f1_score(y_test,y_pred_log),
          f1_score(y_test,y_pred_rf),
          f1_score(y_test,y_pred_svm),
          f1_score(y_test,y_pred_gb),
          f1_score(y_test,y_pred_ada)],
    "ROC-AUC":[roc_auc_score(y_test,log_reg.predict_proba(X_test_scaled)[:,1]),
               roc_auc_score(y_test,rf.predict_proba(X_test)[:,1]),
               roc_auc_score(y_test,svm.predict_proba(X_test_scaled)[:,1]),
               roc_auc_score(y_test,gb.predict_proba(X_test)[:,1]),
               roc_auc_score(y_test,ada.predict_proba(X_test)[:,1])]
}

df_results = pd.DataFrame(results)
print(df_results)


In [ ]:
# Gráfico comparativo
df_results.set_index("Modelo")[["Recall","F1","ROC-AUC"]].plot(kind="bar", figsize=(10,6))
plt.title("Comparação de Modelos Supervisionados")
plt.ylabel("Score")
plt.show()


In [ ]:
agg = AgglomerativeClustering(n_clusters=2)
labels_agg = agg.fit_predict(X_unsupervised_scaled)


### Métricas de Clusterização
Foram utilizadas métricas para avaliar a qualidade dos clusters:
- **Silhouette Score**  
- **Davies-Bouldin Index**  
- **Calinski-Harabasz Index**


In [ ]:
print("K-Means Silhouette:", silhouette_score(X_unsupervised_scaled, labels_kmeans))
print("Agglomerative Silhouette:", silhouette_score(X_unsupervised_scaled, labels_agg))

mask = labels_dbscan != -1
if np.any(mask):
    print("DBSCAN Silhouette:", silhouette_score(X_unsupervised_scaled[mask], labels_dbscan[mask]))
else:
    print("DBSCAN não formou clusters válidos.")


### Avaliação Crítica da Solução

Os modelos supervisionados apresentaram bom desempenho, com destaque para o Random Forest e Gradient Boosting.  
O Recall foi priorizado, garantindo maior identificação de clientes inadimplentes.  

Nos modelos não supervisionados, os clusters revelaram padrões interessantes de segmentação, mas não substituem a previsão direta de inadimplência.  
O uso combinado das duas abordagens amplia o valor estratégico da solução, oferecendo previsões e insights de perfil.



### Conclusão

O projeto demonstrou a viabilidade técnica da aplicação de Machine Learning em risco de crédito.  
A combinação de **modelos supervisionados** (para previsão de inadimplência) e **não supervisionados** (para segmentação de clientes) fornece uma solução robusta e estratégica.  

Como próximos passos, recomenda-se:
- Inclusão de variáveis externas e comportamentais.  
- Testes com outros algoritmos de ensemble (XGBoost, LightGBM).  
- Monitoramento contínuo em produção para detectar data drift.  

Assim, o projeto reforça o potencial do ML para apoiar decisões financeiras com maior precisão e inteligência.

